# Guided Lab: Titanic, Feature Engineering and Visualisation

**Goal:** Work through a complete ML workflow on a dataset from exploration to a simple model.

## What you will do

1. Load and inspect the data
2. Visualize patterns and form hypotheses about useful features
3. Train a baseline model
4. Engineer a new feature and compare results
5. Reflect on what you learned

---
## Part 1: Setup & First Look

Run the cell below to load the Titanic dataset (no download needed).

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = sns.load_dataset("titanic")
df.head()

In [ ]:
df.info()

**Quick check-in**:

1. What is the **target** (label) we want to predict?
2. Name **3 columns** that might help predict it.
3. Which columns look **less useful** and why?
4. What information about the data can you get from the `df.info()` call?

---
## Part 2: Visual Exploration

Good models start with understanding the data. Run the two plots below first, they reveal strong patterns.

After each plot, ask: *Does this column look useful for predicting survival?*

In [ ]:
sns.barplot(data=df, x="sex", y="survived")
plt.title("Survival Rate by Gender")
plt.show()

In [ ]:
sns.barplot(data=df, x="class", y="survived")
plt.title("Survival Rate by Passenger Class")
plt.show()

**Question:** Which gender had a higher survival rate? Which class survived most often? Why might that be?

In [ ]:
# Overall survival balance
sns.countplot(data=df, x="survived")
plt.title("Survival Distribution")
plt.show()

In [ ]:
# Age distribution
sns.histplot(df["age"], bins=20)
plt.title("Age Distribution")
plt.show()

In [ ]:
# Numeric correlations
numeric_df = df.select_dtypes(include=["number"])
plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Try your own plot: change x and y to any columns you are curious about
sns.barplot(data=df, x="embark_town", y="survived")
plt.title("Survival Rate by Embark Town")
plt.show()

**Checkpoint:** Based on what you saw, pick your **top 3 features** for predicting survival and explain why.

---
## Part 3: A Baseline

We will start small, on purpose. Our first model uses just **one** feature: `age`. This gives us a low bar so we can clearly see the effect of adding and engineering features later.

Run the helper cell first, then the baseline cell. If you skipped Part 1, run the setup cell so `df` exists.

The helper handles missing values, scaling, encoding, training, and evaluation so you can focus on the **features**. No need to understand the whole pipeline now (we will get to this).

In [ ]:
def evaluate_model(df, feature_cols, target="survived"):
    """Build a pipeline, train, and return train/test accuracy."""
    X = df[feature_cols]
    y = df[target]

    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

    numeric_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_transformer = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])
    preprocessor = ColumnTransformer([
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ])
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, solver="liblinear")),
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    model.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))

    return {"train": train_acc, "test": test_acc}

In [ ]:
# A "dumb" reference: always predict the most common outcome (everyone dies)
majority_accuracy = max(df["survived"].mean(), 1 - df["survived"].mean())

# Weak baseline: a single feature
weak_features = ["age"]
weak = evaluate_model(df, weak_features)

print(f"Always guess majority class: {majority_accuracy:.3f}")
print(f"Weak baseline (age only), train: {weak['train']:.3f}, test: {weak['test']:.3f}")

**Surprise:** `age` alone scores *worse* than simply guessing that everyone died. A single feature can carry very little signal. This is our starting point. Now let's improve it.

---
## Part 4: Feature Selection & Engineering

Our weak baseline used only `age`. The simplest improvement is **feature selection**: choosing better columns that already exist.

From Part 2 we saw `sex` and `pclass` mattered a lot. Let's add them (plus `fare`) and see what happens.

In [ ]:
# Feature selection: add the columns exploration said were useful
baseline_features = ["age", "sex", "fare", "pclass"]
baseline = evaluate_model(df, baseline_features)

print("Comparison:")
print(f"  Weak baseline (age only):   {weak['test']:.3f}")
print(f"  + sex, fare, pclass:        {baseline['test']:.3f}")

**Big jump!** Just by *selecting* `sex`, `fare`, and `pclass`, accuracy leaps from below the majority-class line to around 0.80. Picking the right existing columns often gives a large boost on its own.

### Now engineer a new feature: family_size

Selection got us a long way. Next we *create* information that isn't directly in the data.

Exploration suggested that *who you traveled with* might matter. Let's combine sibling/spouse and parent/child counts into one feature: **family_size**.

In [ ]:
df["family_size"] = df["sibsp"] + df["parch"] + 1
df[["sibsp", "parch", "family_size"]].head()

**Pause:** Would you expect solo travelers, small families, or large families to survive differently? Why?

In [ ]:
# Visualize survival by family size before retraining
sns.barplot(data=df, x="family_size", y="survived")
plt.title("Survival Rate by Family Size")
plt.show()

In [ ]:
engineered_features = baseline_features + ["family_size"]

with_family = evaluate_model(df, engineered_features)

print(f"With family_size, train: {with_family['train']:.3f}, test: {with_family['test']:.3f}")

In [ ]:
print("Comparison:")
print(f"  Baseline test accuracy:     {baseline['test']:.3f}")
print(f"  + family_size test accuracy: {with_family['test']:.3f}")

### A smarter feature: domain knowledge

`family_size` only nudged accuracy a little. Can we do better by encoding what we *know* about the disaster?

The "women and children first" rule was used when loading lifeboats. Let's turn that historical knowledge into a feature: a passenger is prioritized if they are **female or a child** (age < 16).

This is the heart of feature engineering: combining columns using human insight on top of the raw data.

In [ ]:
# "Women and children first": female OR under 16
df["woman_or_child"] = ((df["sex"] == "female") | (df["age"] < 16)).astype(int)

smart_features = baseline_features + ["family_size", "woman_or_child"]

smart = evaluate_model(df, smart_features)

print("Comparison:")
print(f"  Baseline test accuracy:                  {baseline['test']:.3f}")
print(f"  + family_size test accuracy:             {with_family['test']:.3f}")
print(f"  + family_size + woman_or_child accuracy:  {smart['test']:.3f}")

**Pause:** Combining domain knowledge (`woman_or_child`) with `family_size` gives a bigger lift than either feature alone.

**A word of caution: the `deck` trap.** You might notice that adding `deck` scores even higher. First check how many values are actually present:

```python
df["deck"].notna().sum()  # only ~200 of 891 rows
```

`deck` is missing for about 77% of passengers, so most values are imputed. A feature can look powerful while being mostly guessed. Always check missingness before trusting a gain.

### Add more features

The dataset already has `adult_male` and `alone`. Try adding them. Does test accuracy go up or down?

In [ ]:
# Optional stretch goal
extended_features = engineered_features + ["adult_male", "alone"]

extended = evaluate_model(df, extended_features)

print(f"Extended feature set, train: {extended['train']:.3f}, test: {extended['test']:.3f}")

**Key insight:** Compare **train** vs **test** accuracy above. Train accuracy is almost always higher.

- If train >> test, the model may be **overfitting** (it memorizes the training data and then fails on new data).
- Adding more features does not always help. Sometimes it adds noise.


---
## Part 5: Wrap-Up
Discuss:

1. Which feature seemed most useful in the plots *and* in the model?
2. Did anything surprise you?
3. Why isn't "more features" always better?
4. What would you try next with more time?

**You just completed a full ML workflow:**

Data → Exploration → Features → Model → Evaluation